In [ ]:
import os
import sys
from datetime import datetime, timedelta

project_root = os.path.expanduser('~/git/neoexchange-devel/neoexchange')
sys.path.insert(0, project_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'neox.settings')
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')

import django

django.setup()

os.chdir(project_root)  # temp fix for json issue

from django.contrib.auth.models import User

from core.models import Block, Frame, StaticSource
from core.views import record_block, schedule_submit

## Note ! The code below will cause live submissions to the LCO Observing System - exercise caution !

In [ ]:
try:
    ref_field_type = StaticSource.REFERENCE_FIELD
except AttributeError:
    ref_field_type = 16
coj_fields = StaticSource.objects.filter(source_type=ref_field_type, name__contains='COJ 2026 Field')   # change this to loop over bad fields: 15, 14, 13
site = 'coj'      # or missing fields: 0-5, 7, 10,
username = 'tlister@lcogt.net'
user = User.objects.get(username=username)
proposal = 'LCO2026A-003'
num_exps = 9

# Scheduling when all reference fields are defined (run once per night)

obs_date = datetime(2026, 7, 2)
try_num = 16 # Increment this each day  
for field in coj_fields.order_by('id'):
    print(f"Working on Field #{field.name} for {site.upper()}: ", end='')
    data = {'ra_deg' : field.ra, 'dec_deg' :  field.dec, 'source_id' : field.name,
            'slot_length' : 15,
            'exp_count' : num_exps, 'exp_length' : 60, 'gp_explength' : 60,  'rp_explength' : 60,  'ip_explength' : 60,  'zp_explength' : 60,
            'filter_pattern' : 'gp',
            'site' : site.lower(), 'site_code' : 'E10',
            'start_time' : obs_date, 'end_time' : obs_date+timedelta(seconds=86400-1),
            'user_id' : username, 'group_name' : field.name + f" try{try_num}",
            'proposal_code' : proposal,
            'max_airmass' : 1.5, 'min_lunar_dist' : 40,
            'bin_mode' : None, 'muscat_sync' : False, 'instrument_code' : '', 'period' : None, 'jitter' : None}
    observed_blocks = Block.objects.filter(calibsource=field, num_observed__gte=1)
    good_frames = Frame.objects.filter(block__in=observed_blocks, frametype=Frame.BANZAI_RED_FRAMETYPE).exclude(quality__contains=Frame.QUALITY_STREAKED)
    num_frames = good_frames.count()
    num_good_blocks = good_frames.values('block').distinct().count()
    expected_num_frames = max(num_good_blocks, 1) * num_exps * 4
    print(f"{num_frames}/{expected_num_frames} obtained. Done?:  {num_frames >= expected_num_frames}")
    

    
    if num_frames < expected_num_frames:
        print("Scheduling")
        tracking_num, sched_params = schedule_submit(data, field, data['user_id'])
#        tracking_num = None
    else:
        print("Already observed")
        tracking_num = None
   
    if tracking_num is not None:
        try:
            block_resp = record_block(tracking_num, sched_params, data, field, user)
        except TypeError:
            block_resp = record_block(tracking_num, sched_params, data, field)
        print(f"Created Block {tracking_num:} ? {block_resp:}")
